In [ ]:
# This code is for Google Colab

In [ ]:
# Cell 0: Install dependencies
!pip -q install "transformers>=4.41" "accelerate>=0.30" bitsandbytes \
                 sentence-transformers faiss-cpu pandas numpy requests tqdm


In [ ]:
# Cell 1: Imports & credentials
import os
import json
import re
import numpy as np
import pandas as pd
import requests
import torch
import faiss
from pprint import pprint
from tqdm import tqdm
from google.colab import userdata
!pip -q install google-generativeai
!pip -q install groq

TMDB_BEARER  = userdata.get("TMDB_BEARER")
HF_TOKEN     = userdata.get("HF_TOKEN")
GH_TOKEN = userdata.get("GH_TOKEN")

print("✅ Imports done")


✅ Imports done


In [4]:
# Cell 2: Clone repo
!git clone https://{GH_TOKEN}@github.com/ryuichi-github/ryuichi-github-info290-2026s-group5.git


Cloning into 'ryuichi-github-info290-2026s-group5'...
fatal: could not read Password for 'https://%7BGH_TOKEN%7D@github.com': No such device or address


In [ ]:
# Cell 3: Load dataset
REPO_ROOT = "ryuichi-github-info290-2026s-group5"
file_path = os.path.join(REPO_ROOT, "tmdb-fetch/tmdb_movies.csv")

if os.path.exists(file_path):
    movies_df = pd.read_csv(file_path)
    print(f"✅ Loaded {len(movies_df)} movies")
    print(movies_df.columns.tolist())
else:
    raise FileNotFoundError(f"{file_path} not found. Check repo clone.")


✅ Loaded 4816 movies
['id', 'title', 'tagline', 'overview', 'release_date', 'genre_ids', 'genre_names', 'vote_average', 'vote_count', 'popularity', 'poster_path', 'backdrop_path', 'runtime', 'original_language', 'status', 'revenue', 'budget', 'belongs_to_collection']


In [ ]:
# Cell 4: TMDB API helper + genre mapping
def tmdb_get(url, params=None):
    headers = {
        "Authorization": f"Bearer {TMDB_BEARER}",
        "Content-Type": "application/json;charset=utf-8"
    }
    r = requests.get(url, headers=headers, params=params, timeout=30)
    r.raise_for_status()
    return r.json()

# Verify API connection
tmdb_get("https://api.themoviedb.org/3/authentication")

# Genre id <-> name mapping
genre_data = tmdb_get("https://api.themoviedb.org/3/genre/movie/list", params={"language": "en-US"})
GENRE_ID_TO_NAME = {g["id"]: g["name"] for g in genre_data.get("genres", [])}
GENRE_NAME_TO_ID = {v.lower(): k for k, v in GENRE_ID_TO_NAME.items()}

print(f"✅ Genres loaded: {len(GENRE_ID_TO_NAME)}")


✅ Genres loaded: 19


In [ ]:
# ============================================================
# Pipeline Overview
# ============================================================
# Step 0: Fetch data from TMDB API
#         → tmdb_movies.csv (saved to GitHub, 4,816 movies)
#
# Step 1: Build RAG corpus
#         → Filter to movies with taglines (4,312 movies)
#         → Embed [title + tagline + overview] with Sentence-BERT
#         → Build FAISS IndexFlatIP (cosine similarity)
#
# Step 2: Generation
#         → V1: Zero-shot baseline (no RAG)
#         → V2: RAG (retrieve top-k → pass tagline+overview as reference → generate)
#
# Step 3: Evaluation (TBD)
#         → Human A/B test
#
# Input:  1-sentence movie concept
# Output: overview + tagline + poster_art_direction
# ============================================================


In [ ]:
# Cell 6: Step 1 - Build RAG corpus
# Corpus: item with tagline data (4,312 movies)
# Embedding: title + tagline + overview

from sentence_transformers import SentenceTransformer

# Filter to movies with taglines
corpus_df = movies_df[
    movies_df["tagline"].notna() & (movies_df["tagline"].str.strip() != "")
].copy().reset_index(drop=True)

print(f"Total movies: {len(movies_df)}")
print(f"Corpus (tagline あり): {len(corpus_df)} ({len(corpus_df)/len(movies_df)*100:.1f}%)")
print(f"Excluded (tagline なし): {len(movies_df) - len(corpus_df)}")

# Embed [title + tagline + overview]
embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

corpus_texts = (
    corpus_df["title"] + " " + corpus_df["tagline"] + " " + corpus_df["overview"]
).tolist()

embeddings = embed_model.encode(
    corpus_texts,
    normalize_embeddings=True,
    show_progress_bar=True
).astype("float32")

# Build FAISS index
dim = embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(dim)
faiss_index.add(embeddings)

print(f"✅ Embeddings shape: {embeddings.shape}")
print(f"✅ FAISS index size: {faiss_index.ntotal}")


Total movies: 4816
Corpus (tagline あり): 4312 (89.5%)
Excluded (tagline なし): 504


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/135 [00:00<?, ?it/s]

✅ Embeddings shape: (4312, 384)
✅ FAISS index size: 4312


In [ ]:
# Cell 7: Load LLM (Mistral-7B-Instruct-v0.2, 4-bit quantized)
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    device_map="auto",
    quantization_config=bnb_config,
)

print("✅ Model loaded on:", model.device)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

✅ Model loaded on: cuda:0


# Local Image Captioning Tests - run locally


1.   VIT-GPT2
2.   BLIP_2

Decided to not go with these; they are too heavy and are running on CPU.

In [ ]:
# Test 1: VIT-GPT2
# Cell 8: Local image captioner (vit-gpt2, direct model load)
from transformers import VisionEncoderDecoderModel, ViTImageProcessor, AutoTokenizer as VitTokenizer
from io import BytesIO
from PIL import Image

vit_model = VisionEncoderDecoderModel.from_pretrained(
    "nlpconnect/vit-gpt2-image-captioning"
).to("cpu")  # load on CPU (to preserve memory)
vit_processor = ViTImageProcessor.from_pretrained(
    "nlpconnect/vit-gpt2-image-captioning"
)
vit_tokenizer = VitTokenizer.from_pretrained(
    "nlpconnect/vit-gpt2-image-captioning"
)
vit_model.eval()
print("✅ vit-gpt2 loaded on CPU")

In [ ]:
TMDB_IMAGE_BASE = "https://image.tmdb.org/t/p/w500"

def describe_poster(poster_path: str) -> str:
    if not poster_path or pd.isna(poster_path):
        return ""
    try:
        url = TMDB_IMAGE_BASE + poster_path
        r = requests.get(url, timeout=15, stream=True)
        img_bytes = b"".join(
            chunk for chunk in r.iter_content(chunk_size=4096) if chunk
        )
        img = Image.open(BytesIO(img_bytes)).convert("RGB")

        pixel_values = vit_processor(
            images=img, return_tensors="pt"
        ).pixel_values  # ← no .to("cuda")

        with torch.no_grad():
            output_ids = vit_model.generate(
                pixel_values,
                max_new_tokens=50,
                num_beams=4,
            )

        caption = vit_tokenizer.decode(output_ids[0], skip_special_tokens=True)
        return caption.strip()

    except Exception as e:
        print(f"Poster description failed for {poster_path}: {e}")
        return ""

print("✅ describe_poster ready (vit-gpt2 on CPU)")

In [ ]:
# Cell 10: Testing vit-gpt2 on a single TMDB poster
test_path = "/dldIX0q5jewe8rSyCh8d5I1RYx3.jpg"  # Take Shelter poster
result = describe_poster(test_path)
print(f"Caption: '{result}'")

In [ ]:
# Test 2: Use BLIP2 (with prompt)

In [ ]:
#Set up for BLIP1, using blip2-flan-t5-xl
from transformers import Blip2Processor, Blip2ForConditionalGeneration

blip_processor = Blip2Processor.from_pretrained("Salesforce/blip2-flan-t5-xl")
blip_model = Blip2ForConditionalGeneration.from_pretrained(
    "Salesforce/blip2-flan-t5-xl",
    torch_dtype=torch.float32,
).to("cpu")
blip_model.eval()
print("✅ BLIP-2 FlanT5 loaded on CPU")

In [ ]:
def describe_poster(poster_path: str) -> str:
    if not poster_path or pd.isna(poster_path):
        return ""
    try:
        url = TMDB_IMAGE_BASE + poster_path
        r = requests.get(url, timeout=15, stream=True)
        img_bytes = b"".join(
            chunk for chunk in r.iter_content(chunk_size=4096) if chunk
        )
        img = Image.open(BytesIO(img_bytes)).convert("RGB")

        prompt = """Question: Describe in detail what you see in this movie poster, including people, expressions, background, weather, animals, colors, and mood. Answer:"""
        inputs = blip_processor(img, prompt, return_tensors="pt")

        with torch.no_grad():
            out = blip_model.generate(**inputs, max_new_tokens=75)

        return blip_processor.decode(out[0], skip_special_tokens=True).strip()

    except Exception as e:
        print(f"Poster description failed for {poster_path}: {e}")
        return ""

print("✅ describe_poster ready (BLIP-2)")

In [ ]:
# Prompt format comparison test on Take Shelter poster
test_path = "/dldIX0q5jewe8rSyCh8d5I1RYx3.jpg"

url = TMDB_IMAGE_BASE + test_path
r = requests.get(url, timeout=15, stream=True)
img_bytes = b"".join(chunk for chunk in r.iter_content(chunk_size=4096) if chunk)
img = Image.open(BytesIO(img_bytes)).convert("RGB")

prompts = [
    "Question: Describe in detail the people, weather, animals, colors and mood in this movie poster. Answer:",
    "Describe in detail what you see in this movie poster, including people, expressions, background, weather, animals, colors, and mood:",
    "This movie poster shows",
]

for p in prompts:
    inputs = blip_processor(img, p, return_tensors="pt")
    with torch.no_grad():
        out = blip_model.generate(**inputs, max_new_tokens=60)
    result = blip_processor.decode(out[0], skip_special_tokens=True).strip()
    print(f"Prompt: '{p[:60]}...'")
    print(f"Output: '{result}'")
    print()

In [ ]:
#Test 3: Use BLIP2 for poster captioning (without prompt)

In [ ]:
def describe_poster(poster_path: str) -> str:
    if not poster_path or pd.isna(poster_path):
        return ""
    try:
        url = TMDB_IMAGE_BASE + poster_path
        r = requests.get(url, timeout=15, stream=True)
        img_bytes = b"".join(
            chunk for chunk in r.iter_content(chunk_size=4096) if chunk
        )
        img = Image.open(BytesIO(img_bytes)).convert("RGB")

        # Image only, no prompt
        inputs = blip_processor(images=img, return_tensors="pt")

        with torch.no_grad():
            out = blip_model.generate(**inputs, max_new_tokens=75)

        return blip_processor.decode(out[0], skip_special_tokens=True).strip()

    except Exception as e:
        print(f"Poster description failed for {poster_path}: {e}")
        return ""

In [ ]:
test = describe_poster("/dldIX0q5jewe8rSyCh8d5I1RYx3.jpg")
print(f"BLIP-2 FlanT5 (no prompt): '{test}'")

In [ ]:
v1_blip, v2_blip, topk, refs_blip = run_v1_v2(
    movie_input,
    k=5,
    genre_filter=["thriller", "horror", "mystery"]
)
print("=== References with BLIP-2 ===")
print(refs_blip)
print("\n=== V2 with BLIP-2 ===")
pprint(v2_blip)

# API Captioning Tests


1.   GEMINI FLASH 2.5 (PRO is rate-limited)
2.   GROQ LLAMA 4 SCOUT

Comparing both responses currently!

In [ ]:
# Test 1: Gemini Flash 2.5 VLM via API

In [ ]:
#Set up API call for gemini
import google.generativeai as genai
from PIL import Image
from io import BytesIO

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
genai.configure(api_key=GEMINI_API_KEY)
gemini = genai.GenerativeModel("gemini-2.5-flash")

print("✅ Gemini Flash ready")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


✅ Gemini Flash ready


In [ ]:
TMDB_IMAGE_BASE = "https://image.tmdb.org/t/p/w500"

POSTER_PROMPT = """Analyze this movie poster and write a detailed description in flowing prose. Do not use bullet points, numbers, or lists. Do not invent, assume, or hallucinate any details not visible in the image — only describe what you can literally see.

Your description must cover: the composition and positioning of subjects, the direction and quality of lighting and shadows, the overall color palette including dominant and accent colors, the emotional mood and atmosphere, the typography including text position and how it interacts with the subjects, and one key iconic visual symbol that defines the poster.

Be specific, vivid, and literal. This description will be used by a designer to recreate the visual style for a new movie poster."""

def describe_poster_gemini(poster_path: str) -> str:
    if not poster_path or pd.isna(poster_path):
        return ""
    try:
        url = TMDB_IMAGE_BASE + poster_path
        r = requests.get(url, timeout=15, stream=True)
        img_bytes = b"".join(
            chunk for chunk in r.iter_content(chunk_size=4096) if chunk
        )
        img = Image.open(BytesIO(img_bytes)).convert("RGB")

        response = gemini.generate_content([img, POSTER_PROMPT])
        return response.text.strip()

    except Exception as e:
        print(f"Poster description failed for {poster_path}: {e}")
        return ""

print("✅ describe_poster ready (Gemini Flash)")

✅ describe_poster ready (Gemini Flash)


In [ ]:
test = describe_poster_gemini("/dldIX0q5jewe8rSyCh8d5I1RYx3.jpg")
print(f"Gemini caption: '{test}'")

Gemini caption: 'The movie poster for "Take Shelter" immediately conveys a profound sense of foreboding and unease through its masterful composition and atmospheric elements. Dominating the foreground, a man with a concerned and intense upward gaze occupies the lower left portion of the frame, his features illuminated from the front, highlighting a slightly ruddy complexion and prominent brow. He wears a muted, light tan collared shirt over a white undershirt, presenting a grounded, perhaps blue-collar appearance. Behind him, slightly to the right and receding into the mid-ground, stands a woman with red hair, her expression one of deep worry and apprehension as she also looks upward, clad in a delicate light green or teal garment. Further back and to the right, partially visible, is a young girl with brown hair, gazing to her right with a sad or confused look, wearing a pink top adorned with a subtle floral pattern. This staggered arrangement of the characters suggests a familial unit

In [ ]:
# Test 2: Groq - Llama 4 scout

In [ ]:
#API call for Groq
from groq import Groq
import base64

GROQ_API_KEY = userdata.get("GROQ_API_KEY")
groq_client = Groq(api_key=GROQ_API_KEY)

def describe_poster_groq(poster_path: str) -> str:
    if not poster_path or pd.isna(poster_path):
        return ""
    try:
        url = TMDB_IMAGE_BASE + poster_path
        r = requests.get(url, timeout=15, stream=True)
        img_bytes = b"".join(
            chunk for chunk in r.iter_content(chunk_size=4096) if chunk
        )
        img_b64 = base64.b64encode(img_bytes).decode("utf-8")

        response = groq_client.chat.completions.create(
            model="meta-llama/llama-4-scout-17b-16e-instruct",
            messages=[
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:image/jpeg;base64,{img_b64}"
                            }
                        },
                        {
                            "type": "text",
                            "text": POSTER_PROMPT  # same prompt as Gemini
                        }
                    ]
                }
            ],
            max_tokens=300  # ← increased from 100 for detailed prose
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"Poster description failed: {e}")
        return ""

print("✅ describe_poster_groq ready (Llama 4 Scout)")

✅ describe_poster_groq ready (Llama 4 Scout)


In [ ]:
test = describe_poster_groq("/dldIX0q5jewe8rSyCh8d5I1RYx3.jpg")
print(f"Groq Llama caption: '{test}'")

Groq Llama caption: 'The movie poster for "Take Shelter" presents a haunting and ominous scene. At the top of the image, the title "TAKE SHELTER" is prominently displayed in bold white text, stretching across the frame.

In the foreground, a man with short brown hair is positioned centrally, wearing a light green button-down shirt over a white t-shirt. He gazes upwards to the right with a concerned expression. Behind him, a woman with short red hair and a young girl with long brown hair are partially visible.

The background features a vast expanse of yellow-brown grass, with two tornadoes forming on the horizon to the right. The sky above is dark and foreboding, with swirling clouds. A large flock of black birds flies through the sky, adding to the sense of turmoil.

The overall color palette is characterized by muted tones, with the dominant colors being various shades of brown, yellow, and gray. The lighting appears to be natural, likely depicting a sunset or sunrise, casting a warm

# Running the RAG with all 3 prompts + Example Case

In [ ]:
# Cell 12: Prompt builder + V1/V2 runner

def build_prompt(movie_input: dict, refs: str) -> str:
    premise   = movie_input.get("core_premise", "")
    theme     = movie_input.get("thematic_core", "")
    negatives = movie_input.get("negative_constraints", "")

    return f"""You are a movie marketing generator.

Generate marketing assets for the movie described below.

OUTPUT: ONE JSON object with EXACTLY these keys:
- overview (<= 80 words, compelling promo synopsis)
- tagline (<= 12 words)
- poster_art_direction (<= 60 words)

JSON FORMAT:
{{"overview": "", "tagline": "", "poster_art_direction": ""}}

MOVIE DETAILS:
Core Premise: {premise}
Thematic Core: {theme}
IMPORTANT - Do NOT do the following: {negatives}

REFERENCES from similar successful movies (use for tone and syntax ONLY; do NOT copy plot):
{refs}

CONSTRAINTS:
- Output ONLY the JSON object (no markdown, no backticks, no commentary).
- tagline must be original and specific to this movie, not generic.
- overview must be written as compelling promo copy, not a plot summary.
- End immediately after the final '}}'.

Now output the JSON:
"""


def run_v1_v2(movie_input: dict, k: int = 5, genre_filter=None):
    query = movie_input["core_premise"]

    # V1: no RAG
    p1 = build_prompt(movie_input, refs="(none)")
    t1 = generate_text(p1)
    j1 = extract_best_json(t1)

    # V2: RAG with visual references
    topk = retrieve(query, k=k, genre_filter=genre_filter)
    refs = refs_to_text(topk, n=k, use_visual=True)
    p2 = build_prompt(movie_input, refs=refs)
    t2 = generate_text(p2)
    j2 = extract_best_json(t2)

    return j1, j2, topk, refs

print("✅ Prompt builder and runner ready")


✅ Prompt builder and runner ready


In [ ]:
# Cell 11: Core functions - Retrieval, Generation, JSON extraction

def _normalize_genre_filter(genre_filter):
    if genre_filter is None:
        return None
    if isinstance(genre_filter, (list, tuple)) and len(genre_filter) > 0:
        if isinstance(genre_filter[0], str):
            ids = [GENRE_NAME_TO_ID[name.lower()] for name in genre_filter
                   if name.lower() in GENRE_NAME_TO_ID]
            return ids if ids else []
        if isinstance(genre_filter[0], int):
            return list(genre_filter)
    raise ValueError("genre_filter must be None, list of genre names, or list of genre ids.")


def retrieve(query: str, k: int = 5, genre_filter=None) -> pd.DataFrame:
    q = embed_model.encode([query], normalize_embeddings=True).astype("float32")
    gf_ids = _normalize_genre_filter(genre_filter)

    if gf_ids is None:
        scores, idx = faiss_index.search(q, k)
        out = corpus_df.iloc[idx[0]].copy()
        out["score"] = scores[0]
        return out.reset_index(drop=True)

    gf_ids = set(gf_ids)
    mask = corpus_df["genre_ids"].apply(
        lambda gids: any(gid in gf_ids for gid in eval(gids))
    ).to_numpy()
    candidate_idx = np.where(mask)[0]

    if candidate_idx.size == 0:
        scores, idx = faiss_index.search(q, k)
        out = corpus_df.iloc[idx[0]].copy()
        out["score"] = scores[0]
        out["note"] = "No genre-filtered candidates; fell back to unfiltered."
        return out.reset_index(drop=True)

    cand_emb = embeddings[candidate_idx]
    scores = cand_emb @ q[0]
    topk_local = np.argsort(-scores)[:k]
    topk_idx = candidate_idx[topk_local]
    out = corpus_df.iloc[topk_idx].copy()
    out["score"] = scores[topk_local]
    return out.reset_index(drop=True)


def generate_text(prompt: str, max_new_tokens: int = 320) -> str:
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.6,
            top_p=0.9,
            repetition_penalty=1.10,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0], skip_special_tokens=True)


REQUIRED_KEYS = {"overview", "tagline", "poster_art_direction"}

def extract_best_json(text: str) -> dict:
    decoder = json.JSONDecoder()
    objs = []
    i = 0
    while True:
        start = text.find("{", i)
        if start == -1:
            break
        try:
            obj, end = decoder.raw_decode(text[start:])
            objs.append(obj)
            i = start + end
        except json.JSONDecodeError:
            i = start + 1
    if not objs:
        raise ValueError("No JSON object found in model output.")

    def score(obj):
        if not isinstance(obj, dict):
            return -10
        if not REQUIRED_KEYS.issubset(set(obj.keys())):
            return -5
        s = 0
        tagline  = obj.get("tagline", "")
        poster   = obj.get("poster_art_direction", "")
        overview = obj.get("overview", "")
        if isinstance(tagline, str) and len(tagline.strip()) > 0:
            s += 2
            if len(tagline.split()) <= 14:
                s += 1
        if isinstance(poster, str) and len(poster.strip()) > 20:
            s += 2
        if isinstance(overview, str) and len(overview.strip()) > 40:
            s += 2
        return s

    return max(objs, key=score)


def refs_to_text(df: pd.DataFrame, n: int = 5, use_visual: bool = True) -> str:
    lines = []
    for _, r in df.head(n).iterrows():
        if pd.notna(r["tagline"]) and r["tagline"].strip():
            ov = (r["overview"] or "")[:200].replace("\n", " ")
            line = f'- {r["title"]}: tagline: "{r["tagline"]}" | overview: "{ov}"'
            if use_visual:
                visual = describe_poster(r.get("poster_path", None))
                if visual:
                    line += f' | visual style: "{visual}"'
            lines.append(line)
    return "\n".join(lines)

print("✅ Core functions ready")

✅ Core functions ready


In [ ]:
movie_input = {
    "core_premise": "A low-budget psychological thriller set in a rainy rural village, where the only clinic is cut off by flooding.",
    "thematic_core": "Paranoia and distrust between strangers forced into prolonged isolation.",
    "negative_constraints": "Do NOT invent characters, disappearances, or a villain. The flooding is setting only — not a thriller device. Do NOT use the phrase 'Isolation breeds suspicion' or any generic isolation tagline.",
}

# V1 + V2 with Groq
print("Running V2 with Groq Llama...")
def describe_poster(poster_path):
    return describe_poster_groq(poster_path)

v1, v2_groq, topk, refs_groq = run_v1_v2(  # ← capture v1 and topk here
    movie_input, k=5, genre_filter=["thriller", "horror", "mystery"]
)

# V2 with Gemini
print("Running V2 with Gemini 2.5 Flash...")
def describe_poster(poster_path):
    return describe_poster_gemini(poster_path)

_, v2_gemini, _, refs_gemini = run_v1_v2(
    movie_input, k=5, genre_filter=["thriller", "horror", "mystery"]
)

# Retrieved movies
print("\n=== Retrieved movies ===")
display(topk[["title", "genre_names", "tagline", "score"]])

# References
print("\n=== References: Groq Llama ===")
print(refs_groq)

print("\n=== References: Gemini 2.5 Flash ===")
print(refs_gemini)

# Final comparison
print("\n" + "="*60)
print("FINAL COMPARISON")
print("="*60)

print("\n--- V1: Zero-Shot (no visual) ---")
pprint(v1)

print("\n--- V2: RAG + Groq Llama 4 Scout captions ---")
pprint(v2_groq)

print("\n--- V2: RAG + Gemini 2.5 Flash captions ---")
pprint(v2_gemini)

Running V2 with Groq Llama...
Running V2 with Gemini 2.5 Flash...

=== Retrieved movies ===


,title,genre_names,tagline,score
0,Take Shelter,"['Thriller', 'Drama', 'Horror']",Far away from the cruel world.,0.489664
1,Hard Rain,"['Thriller', 'Crime', 'Action']",A simple plan. An instant fortune. Just add wa...,0.457851
2,Cure,"['Crime', 'Thriller', 'Horror', 'Mystery']",Madness. Terror. Murder.,0.427415
3,Regression,"['Horror', 'Mystery', 'Thriller']",Fear always finds its victim,0.415955
4,Delirium,"['Horror', 'Thriller']",It's all in your head,0.386188



=== References: Groq Llama ===
- Take Shelter: tagline: "Far away from the cruel world." | overview: "Plagued by a series of apocalyptic visions, a young husband and father questions whether to shelter his family from a coming storm, or from himself." | visual style: "The movie poster for "Take Shelter" presents a haunting and ominous visual narrative. At the top, the title "TAKE SHELTER" is emblazoned in bold white letters, set against a backdrop of a dark and foreboding sky teeming with a large flock of birds in flight.

In the foreground, three individuals are positioned in a layered formation. The man at the front has short brown hair and wears an olive-green collared shirt over a white T-shirt, while the woman behind him sports short red hair and a light green shirt. A young girl with long brown hair peeks out from behind them, adding a sense of vulnerability to the scene.

The trio's gazes are directed upwards and to the right, their faces bathed in a warm glow that contrasts wi

In [ ]:
import shutil
shutil.copy(
    "/content/data_pipeline_Mar_27.ipynb",  # source — may not exist
    "/content/ryuichi-github-info290-2026s-group5/data_pipeline_Mar_27.ipynb"
)

FileNotFoundError: [Errno 2] No such file or directory: '/content/data_pipeline_Mar_27.ipynb'